# Video → 3D (VGGT on Colab)

Creates **two outputs from your uploaded video**:

| File | What you should see |
|------|---------------------|
| `scene.glb` | Colored 3D point cloud of the place in your video (open in [gltf-viewer](https://gltf-viewer.donmccurdy.com/)) |
| `flythrough.mp4` | Smooth camera move through that same colored 3D |

## Run order
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Step 0**
3. **Runtime → Restart session**
4. Run **Step 1 → Step 7** (skip Step 0 after restart)


## Step 0 — Install (then restart session)


In [ ]:
import os, sys, shutil
SRC = "/content/vggt_src"
if os.path.isdir("/content/vggt") and not os.path.isfile("/content/vggt/models/vggt.py"):
    shutil.rmtree("/content/vggt")
if not os.path.isdir(os.path.join(SRC, "vggt", "models")):
    !git clone --depth 1 https://github.com/facebookresearch/vggt.git {SRC}
!{sys.executable} -m pip -q uninstall -y numpy
!{sys.executable} -m pip -q install "numpy==1.26.4"
!{sys.executable} -m pip -q install Pillow huggingface_hub einops safetensors opencv-python-headless trimesh matplotlib scipy tqdm imageio imageio-ffmpeg
!{sys.executable} -m pip -q install -e {SRC}
print("Install done. Runtime -> Restart session, then Step 1.")


## Step 1 — Imports (after restart)


In [ ]:
import sys
SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import numpy as np, torch
assert torch.cuda.is_available(), "Enable T4 GPU first"
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
print("OK", np.__version__, torch.__version__)


## Step 2 — Upload video


In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print("Video:", VIDEO_PATH)


## Step 3 — Extract frames (~2 fps, max 60 — denser = better looking 3D)


In [ ]:
import cv2
from pathlib import Path
frames_dir = Path("/content/frames")
frames_dir.mkdir(exist_ok=True)
for p in frames_dir.glob("*"):
    p.unlink()
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
# denser sampling for nicer reconstruction
target_fps = 2.0
interval = max(1, int(round(fps / target_fps)))
max_frames = 60
idx = saved = 0
while saved < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    if idx % interval == 0:
        cv2.imwrite(str(frames_dir / f"{saved:06d}.jpg"), frame)
        saved += 1
    idx += 1
cap.release()
image_names = sorted(str(p) for p in frames_dir.glob("*.jpg"))
print(f"frames={len(image_names)} interval={interval}")
assert len(image_names) >= 3, "Need a longer / clearer indoor clip"


## Step 4 — Reconstruct


In [ ]:
import sys, numpy as np, torch
SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    if x.ndim >= 1 and x.shape[0] == 1:
        x = x.reshape(x.shape[1:])
    return x

device = "cuda"
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device).eval()
images = load_and_preprocess_images(image_names).to(device)
print("images", images.shape)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        predictions = model(images)

pose = predictions["pose_enc"]
if pose.ndim == 2:
    pose = pose.unsqueeze(0)
predictions["pose_enc"] = pose
extrinsic, intrinsic = pose_encoding_to_extri_intri(pose, images.shape[-2:])
predictions["extrinsic"] = extrinsic
predictions["intrinsic"] = intrinsic
predictions["images"] = images
for k, v in list(predictions.items()):
    if isinstance(v, torch.Tensor):
        predictions[k] = to_numpy(v)
predictions["pose_enc_list"] = None
predictions["world_points_from_depth"] = unproject_depth_map_to_point_map(
    predictions["depth"], predictions["extrinsic"], predictions["intrinsic"]
)
print("extrinsic", predictions["extrinsic"].shape, "done")


## Step 5 — Export `scene.glb`

Tips for viewing:
- Open in https://gltf-viewer.donmccurdy.com/
- You should see a **colored point cloud**, not only lines
- Camera frustums are turned **OFF** here so lines do not dominate


In [ ]:
import sys
SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from visual_util import predictions_to_glb

# Depth branch is often cleaner for rooms; no camera wireframes
scene = predictions_to_glb(
    predictions,
    conf_thres=25.0,              # keep more confident points (lower = denser)
    filter_by_frames="all",
    show_cam=False,               # IMPORTANT: hides the "lots of lines"
    prediction_mode="Predicted Depthmap",  # depth-unprojected points
)
scene.export("/content/scene.glb")
print("Wrote /content/scene.glb — open in https://gltf-viewer.donmccurdy.com/")


## Step 6 — Export colored `flythrough.mp4`

Previous version drew tiny gray dots (looked like a constellation).
This version:
- uses **RGB colors from your video frames**
- draws **larger points**
- follows the reconstructed camera path


In [ ]:
import numpy as np
import cv2

# ---- build colored point cloud from depth + images ----
depth = np.asarray(predictions["depth"])
if depth.ndim == 4 and depth.shape[-1] == 1:
    depth = depth[..., 0]  # (S,H,W)
extrinsics = np.asarray(predictions["extrinsic"])  # (S,3,4)
intrinsics = np.asarray(predictions["intrinsic"])  # (S,3,3)
imgs = np.asarray(predictions["images"])          # (S,3,H,W) in 0..1
if imgs.ndim != 4:
    raise RuntimeError(f"Unexpected images shape {imgs.shape}")
S, _, H, W = imgs.shape
imgs_hwc = np.transpose(imgs, (0, 2, 3, 1))  # (S,H,W,3)

# subsample pixels for speed
stride = 2
ys = np.arange(0, H, stride)
xs = np.arange(0, W, stride)
grid_x, grid_y = np.meshgrid(xs, ys)
grid_x = grid_x.reshape(-1)
grid_y = grid_y.reshape(-1)

all_pts = []
all_cols = []
conf = predictions.get("depth_conf")
if conf is not None:
    conf = np.asarray(conf)
    if conf.ndim == 4 and conf.shape[-1] == 1:
        conf = conf[..., 0]

for i in range(S):
    z = depth[i, grid_y, grid_x].astype(np.float64)
    valid = z > 1e-4
    if conf is not None:
        c = conf[i, grid_y, grid_x]
        thr = np.percentile(c, 40)
        valid &= c >= thr
    if valid.sum() < 10:
        continue
    u = grid_x[valid].astype(np.float64)
    v = grid_y[valid].astype(np.float64)
    z = z[valid]
    K = intrinsics[i]
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    # pixel -> camera
    Xc = np.stack([(u - cx) / fx * z, (v - cy) / fy * z, z], axis=1)
    # camera -> world  (E is world-to-camera [R|t], so Xc = R Xw + t => Xw = R^T (Xc - t))
    R = extrinsics[i, :, :3]
    t = extrinsics[i, :, 3]
    Xw = (R.T @ (Xc - t).T).T
    col = imgs_hwc[i, grid_y[valid], grid_x[valid], :]
    col = np.clip(col, 0, 1)
    all_pts.append(Xw)
    all_cols.append(col)

pts = np.concatenate(all_pts, axis=0)
cols = np.concatenate(all_cols, axis=0)
print("colored points:", pts.shape[0])

# keep a manageable set
if len(pts) > 250000:
    sel = np.random.default_rng(0).choice(len(pts), 250000, replace=False)
    pts, cols = pts[sel], cols[sel]

center = np.median(pts, axis=0)
scale = np.percentile(np.linalg.norm(pts - center, axis=1), 90) + 1e-6
pts_n = (pts - center) / scale

out_w, out_h = 960, 540
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter("/content/flythrough.mp4", fourcc, 14.0, (out_w, out_h))
point_radius = 2  # larger = less "constellation"


def render(E):
    R, t = E[:, :3], E[:, 3]
    tn = (R @ center + t) / scale
    Xc = (R @ pts_n.T).T + tn
    z = Xc[:, 2]
    valid = z > 0.05
    frame = np.full((out_h, out_w, 3), 30, np.uint8)
    if valid.sum() < 100:
        return frame
    # use mean intrinsics-ish focal for projection in normalized space
    # map with robust percentiles for framing
    u = Xc[valid, 0] / z[valid]
    v = Xc[valid, 1] / z[valid]
    u0, u1 = np.percentile(u, [8, 92])
    v0, v1 = np.percentile(v, [8, 92])
    if u1 - u0 < 1e-3: u1 = u0 + 1e-3
    if v1 - v0 < 1e-3: v1 = v0 + 1e-3
    px = ((u - u0) / (u1 - u0) * (out_w - 1)).astype(np.int32)
    py = ((v - v0) / (v1 - v0) * (out_h - 1)).astype(np.int32)
    # near points on top
    order = np.argsort(-z[valid])
    px, py = px[order], py[order]
    rgb = (cols[valid][order] * 255).astype(np.uint8)
    # depth fade already handled by draw order; draw as small filled circles
    for x, y, color in zip(px[::2], py[::2], rgb[::2]):  # stride for speed
        if 0 <= x < out_w and 0 <= y < out_h:
            cv2.circle(frame, (int(x), int(y)), point_radius, (int(color[2]), int(color[1]), int(color[0])), -1)
    return frame

# interpolate cameras for smoother motion
for i in range(len(extrinsics) - 1):
    for a in np.linspace(0, 1, 5, endpoint=False):
        E = (1 - a) * extrinsics[i] + a * extrinsics[i + 1]
        writer.write(render(E))
writer.write(render(extrinsics[-1]))
writer.release()
print("Wrote /content/flythrough.mp4")


## Step 7 — Download both


In [ ]:
from google.colab import files
from pathlib import Path
assert Path("/content/scene.glb").is_file()
assert Path("/content/flythrough.mp4").is_file()
files.download("/content/scene.glb")
files.download("/content/flythrough.mp4")
print("Attach both in Streamlit. View GLB at https://gltf-viewer.donmccurdy.com/")
